# TEP reactor-pressure candidate study

Generated from `tep_researcher_input.ipynb`. This notebook implements the authored mechanics without adding persistence, merging, or completeness rules. Its objects are **candidate intervals**, not validated complete physical episodes.

In [ ]:
import featuregraph as fg
import pandas as pd

FAULT_NUMBER = 2
SIMULATION_RUN = 10
SIGNAL = "reactor_pressure"
SMOOTH_WINDOW_MINUTES = 20
RATE_EPS = 0.75


def load_run(fault_number=FAULT_NUMBER, simulation_run=SIMULATION_RUN):
    """Load one Tennessee Eastman Process run without altering source values."""
    return fg.datasets.eastman(
        fault_number=fault_number,
        simulation_run=simulation_run,
    ).copy()


def construct_observations(
    source,
    smooth_window_minutes=SMOOTH_WINDOW_MINUTES,
    rate_eps=RATE_EPS,
):
    """Apply exactly the preprocessing, state, event, and identity rules."""
    df = source.copy()
    original_pressure = df[SIGNAL].copy(deep=True)

    df["sample_index"] = df.index
    df["time_hours"] = df["time_(h)"]
    df["reactor_pressure_raw"] = df[SIGNAL]

    time_step_hours = df["time_hours"].diff()
    sample_interval_minutes = time_step_hours.median() * 60
    smooth_window_samples = round(
        smooth_window_minutes / sample_interval_minutes
    )

    if not pd.notna(sample_interval_minutes) or sample_interval_minutes <= 0:
        raise ValueError("The inferred sample interval must be finite and positive.")
    if smooth_window_samples < 1:
        raise ValueError("The smoothing window must contain at least one sample.")
    if not time_step_hours.dropna().gt(0).all():
        raise ValueError("time_hours must be strictly increasing.")

    df["reactor_pressure_smooth"] = (
        df["reactor_pressure_raw"]
        .rolling(
            window=smooth_window_samples,
            min_periods=smooth_window_samples,
            center=True,
        )
        .mean()
    )
    df["reactor_pressure_change"] = df["reactor_pressure_smooth"].diff()
    df["reactor_pressure_rate"] = (
        df["reactor_pressure_change"] / df["time_hours"].diff()
    )
    df["reactor_pressure_valid"] = (
        df["reactor_pressure_smooth"].notna()
        & df["reactor_pressure_rate"].notna()
    )

    valid = df["reactor_pressure_valid"]
    df["reactor_pressure_rising"] = (
        valid & df["reactor_pressure_rate"].gt(rate_eps)
    )
    df["reactor_pressure_falling"] = (
        valid & df["reactor_pressure_rate"].lt(-rate_eps)
    )
    df["reactor_pressure_inactive"] = (
        valid & df["reactor_pressure_rate"].abs().le(rate_eps)
    )

    rising_int = df["reactor_pressure_rising"].astype(int)
    df["enter_reactor_pressure_rising"] = rising_int.diff().eq(1)
    df["exit_reactor_pressure_rising"] = rising_int.diff().eq(-1)
    df["exit_reactor_pressure_rising_id"] = (
        df["exit_reactor_pressure_rising"].cumsum()
    )

    provenance = {
        "fault_number": FAULT_NUMBER,
        "simulation_run": SIMULATION_RUN,
        "signal": SIGNAL,
        "sample_interval_minutes": float(sample_interval_minutes),
        "smooth_window_minutes": smooth_window_minutes,
        "smooth_window_samples": smooth_window_samples,
        "rate_eps_pressure_units_per_hour": rate_eps,
    }
    return df, original_pressure, provenance


def summarize_candidates(df):
    """Construct the candidate-level table specified by the researcher."""
    summary = (
        df.groupby("exit_reactor_pressure_rising_id", sort=True)
        .agg(
            start_index=("sample_index", "min"),
            end_index=("sample_index", "max"),
            start_time_hours=("time_hours", "min"),
            end_time_hours=("time_hours", "max"),
            reactor_pressure_rising_samples=("reactor_pressure_rising", "sum"),
            reactor_pressure_falling_samples=("reactor_pressure_falling", "sum"),
            reactor_pressure_inactive_samples=("reactor_pressure_inactive", "sum"),
            enter_rising_count=("enter_reactor_pressure_rising", "sum"),
            exit_rising_count=("exit_reactor_pressure_rising", "sum"),
            observation_count=("sample_index", "size"),
        )
        .reset_index()
        .rename(columns={"exit_reactor_pressure_rising_id": "candidate_id"})
    )
    summary["duration_hours"] = (
        summary["end_time_hours"] - summary["start_time_hours"]
    )
    summary["candidate_label"] = summary["candidate_id"].map(
        lambda value: f"TEP-{SIMULATION_RUN:02d}-{int(value):03d}"
    )
    summary["boundary_fragment"] = False
    if not summary.empty:
        summary.loc[summary.index[[0, -1]], "boundary_fragment"] = True
    summary["scientific_status"] = "candidate_interval"
    return summary


def validate_study(source, df, original_pressure, summary):
    """Return explicit structural checks; raise if any required check fails."""
    valid = df["reactor_pressure_valid"]
    state_count = df[
        [
            "reactor_pressure_rising",
            "reactor_pressure_falling",
            "reactor_pressure_inactive",
        ]
    ].sum(axis=1)
    rising_int = df["reactor_pressure_rising"].astype(int)

    checks = {
        "time_strictly_increasing": df["time_hours"].diff().dropna().gt(0).all(),
        "raw_pressure_preserved": source[SIGNAL].equals(original_pressure),
        "one_state_per_valid_sample": state_count[valid].eq(1).all(),
        "no_state_on_invalid_sample": state_count[~valid].eq(0).all(),
        "enter_events_match_definition": df[
            "enter_reactor_pressure_rising"
        ].equals(rising_int.diff().eq(1)),
        "exit_events_match_definition": df[
            "exit_reactor_pressure_rising"
        ].equals(rising_int.diff().eq(-1)),
        "candidate_rows_cover_source": int(summary["observation_count"].sum()) == len(df),
        "boundary_fragments_retained": (
            summary.empty or int(summary["boundary_fragment"].sum()) in (1, 2)
        ),
    }
    report = pd.Series(checks, name="passed").rename_axis("check").to_frame()
    failed = report.index[~report["passed"]].tolist()
    if failed:
        raise AssertionError(f"Study validation failed: {failed}")
    return report


In [ ]:
source = load_run()
observations, original_pressure, provenance = construct_observations(source)
candidate_summary = summarize_candidates(observations)
validation_report = validate_study(
    source,
    observations,
    original_pressure,
    candidate_summary,
)

provenance


In [ ]:
fragmentation_diagnostics = pd.Series(
    {
        "source_observations": len(observations),
        "valid_observations": int(observations["reactor_pressure_valid"].sum()),
        "candidate_intervals": len(candidate_summary),
        "enter_rising_events": int(
            observations["enter_reactor_pressure_rising"].sum()
        ),
        "exit_rising_events": int(
            observations["exit_reactor_pressure_rising"].sum()
        ),
        "boundary_fragments_retained": int(
            candidate_summary["boundary_fragment"].sum()
        ),
    },
    name="value",
).rename_axis("diagnostic").to_frame()

display(validation_report)
display(fragmentation_diagnostics)
display(candidate_summary.head(20))


## Interpretation limits

The table above exposes the consequences of the authored threshold and smoothing choices. Frequent candidates are evidence of fragmentation under these rules; they are not permission to add persistence, hysteresis, merging, or a completeness criterion. Those remain unresolved scientific choices in the researcher input.